# Ceftriaxone — Phase-1 Species-Mask Diagnostic

**Is the weak species mask (top-500 ≈ 70% of species importance) an artifact of the RF's default `max_features="sqrt"`?**

Sweeps the per-site species RF over 3 configs and reports, per config:
- OOB accuracy (species classification)
- importance concentration at top-{100, 500, 1000}
- mask sizes (union / majority) + pairwise overlap
- masking-drop: `OOB_full − OOB_masked` with the top-500 bins zeroed

Reuses the exact 06-03c data pipeline (same split, downsampling, preprocessing, SEED).

In [ ]:
# ── CONFIG ──
MASK_TOP_K = 500
K_LIST = [100, 500, 1000]
RUN_NAME = "06-03c-Phase1-Mask-Diagnostic"
TARGET_RUN = "01-Run"


In [ ]:
!pip install "flwr[simulation]" maldideepkit maldiamrkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)


In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"

PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06b-Ceftriaxone-E-coli" / RUN_NAME
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Drug: {DRUG_NAME}  |  Run: {RUN_DIR}")


In [ ]:
# ── Load Ceftriaxone — ALL species ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples ({n_sp} species)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")


In [ ]:
# ── Per-site species-stratified 90/10 split ──
client_train = {}; client_test = {}
species_train = {}; species_test = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    n = len(y); idx = np.arange(n).reshape(-1,1)
    itr, iv, _, _ = stratified_species_drug_split(idx, y, species=sp, test_size=0.10, random_state=SEED)
    itr = itr.flatten().astype(int); iv = iv.flatten().astype(int)
    client_train[site] = (X[itr], y[itr]); client_test[site] = (X[iv], y[iv])
    species_train[site] = sp[itr]; species_test[site] = sp[iv]
    print(f"  Site {site}: train={len(itr)} test={len(iv)} species={len(np.unique(sp[itr]))}")


In [ ]:
# ── Downsample training data (keep all R, cap S in bad-ratio species) ──
TRAIN_DS_S_PER_R = 10   # global: all training data (bad-ratio species only)

def downsample_keep_idx(y, sp, s_per_r, bad_ratio=0.10, min_n=400, rng=None):
    rng = rng if rng is not None else np.random.default_rng(SEED)
    idx = np.arange(len(y))
    keep = idx[y == 1].tolist()
    for spec in np.unique(sp):
        m = sp == spec
        n_r = int((m & (y == 1)).sum()); n_s = int((m & (y == 0)).sum())
        total = n_r + n_s
        s_pos = idx[m & (y == 0)]
        if n_r > 0 and total > min_n and (n_r / total) < bad_ratio and n_s > n_r * s_per_r:
            keep.extend(rng.choice(s_pos, size=int(round(n_r * s_per_r)), replace=False).tolist())
        else:
            keep.extend(s_pos.tolist())
    return np.sort(np.array(keep, dtype=int))

_rng_ds = np.random.default_rng(SEED)
for site in SITE_ORDER:
    X, y = client_train[site]; sp = species_train[site]
    keep = downsample_keep_idx(y, sp, TRAIN_DS_S_PER_R, rng=_rng_ds)
    client_train[site] = (X[keep], y[keep]); species_train[site] = sp[keep]
    print(f"  Site {site}: kept {len(keep)}/{len(y)} train")


In [ ]:
# ── Per-site preprocessing ──
client_train_pp = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
print("Per-site preprocessing done.")


In [ ]:
# ── Species RF config sweep + mask concentration + masking-drop ──
CONFIGS = {
    "baseline": dict(n_estimators=300, max_depth=20, min_samples_leaf=5),            # max_features='sqrt' (current)
    "mf_0.2":   dict(n_estimators=300, max_depth=20, min_samples_leaf=5, max_features=0.2),
    "mf_0.5":   dict(n_estimators=500, max_depth=20, min_samples_leaf=5, max_features=0.5),
}

rows = []
overlap_rows = []
for cfg_name, cfg in CONFIGS.items():
    print(f"\n=== {cfg_name}: {cfg} ===")
    site_masks = {}
    for site in SITE_ORDER:
        X_tr, y_tr = client_train_pp[site]
        sp_tr = species_train[site]
        n_sp = len(np.unique(sp_tr))
        if n_sp < 3:
            print(f"  Site {site}: only {n_sp} species — skip")
            site_masks[site] = np.array([], dtype=int)
            continue

        rf = RandomForestClassifier(**cfg, n_jobs=-1, oob_score=True, random_state=SEED)
        rf.fit(X_tr, sp_tr)
        imp = rf.feature_importances_
        top_k = np.argsort(imp)[-MASK_TOP_K:]; top_k.sort()
        site_masks[site] = top_k

        # masking-drop: retrain with top-500 bins zeroed
        X_masked = X_tr.copy(); X_masked[:, top_k] = 0.0
        rf_masked = RandomForestClassifier(**cfg, n_jobs=-1, oob_score=True, random_state=SEED)
        rf_masked.fit(X_masked, sp_tr)

        row = {"config": cfg_name, "site": site, "n_species": n_sp,
               "oob_full": rf.oob_score_, "oob_masked": rf_masked.oob_score_,
               "oob_drop": rf.oob_score_ - rf_masked.oob_score_}
        for K in K_LIST:
            row[f"top{K}_conc"] = imp[np.argsort(imp)[-K:]].sum()
        rows.append(row)
        print(f"  {site}: {n_sp} sp, OOB={rf.oob_score_:.4f}, drop={row['oob_drop']:+.4f}, "
              + ", ".join(f"top{K}={row[f'top{K}_conc']:.1%}" for K in K_LIST))

    # masks + overlap for this config
    all_bins = [site_masks[s] for s in SITE_ORDER if len(site_masks[s]) > 0]
    union = np.unique(np.concatenate(all_bins)) if all_bins else np.array([], dtype=int)
    bc = {}
    for bins in all_bins:
        for b in bins:
            bc[b] = bc.get(b, 0) + 1
    majority = np.array([b for b, c in bc.items() if c >= 2], dtype=int)
    print(f"  union={len(union)}  majority={len(majority)}")
    sites_w = [s for s in SITE_ORDER if len(site_masks[s]) > 0]
    for i, sa in enumerate(sites_w):
        for j, sb in enumerate(sites_w):
            if j <= i: continue
            ov = len(set(site_masks[sa]) & set(site_masks[sb]))
            overlap_rows.append({"config": cfg_name, "pair": f"{sa}∩{sb}", "overlap": ov / MASK_TOP_K})

df_diag = pd.DataFrame(rows)
df_diag.to_csv(OUT_DIR / "mask_diagnostic.csv", index=False)

print("\n===== Mean across sites (per config) =====")
summary_cols = ["oob_full", "oob_drop"] + [f"top{K}_conc" for K in K_LIST]
print(df_diag.groupby("config")[summary_cols].mean().to_string())

if overlap_rows:
    df_ov = pd.DataFrame(overlap_rows)
    print("\n===== Mean pairwise overlap (per config) =====")
    print(df_ov.groupby("config")["overlap"].mean().to_string())

# plot: concentration vs K, per config (mean across sites)
fig, ax = plt.subplots(figsize=(8, 5))
for cfg_name in CONFIGS:
    sub = df_diag[df_diag["config"] == cfg_name]
    y = [sub[f"top{K}_conc"].mean() for K in K_LIST]
    ax.plot(K_LIST, y, marker='o', label=cfg_name)
ax.set_xlabel("Top-K bins"); ax.set_ylabel("Species importance captured")
ax.set_title(f"{DRUG_NAME} — species-mask concentration vs RF config")
ax.legend(); ax.grid(True, ls='--', alpha=0.5)
plt.tight_layout(); plt.savefig(OUT_DIR / "mask_concentration.pdf", bbox_inches="tight"); plt.show()


---
**Done.** See `mask_diagnostic.csv` and `mask_concentration.pdf` in `results/`.

Interpretation:
- Higher `top-K_conc` = more concentrated species importance (better mask).
- Larger `oob_drop` = the top-500 bins carry more species signal.
- If `mf_0.5` lifts concentration toward ~90%+ or changes overlap, the weak mask was an RF artifact → re-run 06-03c with that config. If it stays ~70% with stable overlap, the `none` result is robust.
